## Day 1: Why a Plain LLM Can't Answer From Your Handbook

A plain LLM predicts the next token from patterns learned during training --
it has no mechanism to look anything up, so a question about a private,
never-published document produces either a hallucinated guess or (if we're
lucky) a refusal. RAG fixes this by handing the model the relevant passage
*at answer time*, the same way an open-book exam beats working from memory.

This cell contrasts an ungrounded ("closed-book") answer against a grounded
("open-book") one for the same question -- both are illustrative stand-ins
for real LLM calls (no network access in this environment), but the shape
is exactly what Day 4's real `answer()` function produces.


In [ ]:
handbook_sections = [
    {"id": "sec-4.1", "text": "The office is closed on all federal holidays.", "page": 13},
    {"id": "sec-4.2", "text": "New hires accrue 12 vacation days in their first year.", "page": 14},
    {"id": "sec-4.3", "text": "Employees may take a scenic vacation to the mountains.", "page": 14},
    {"id": "sec-5.1", "text": "Paid time off requests must be submitted two weeks in advance.", "page": 18},
]

def ungrounded_answer(question: str) -> str:
    # Stands in for a plain LLM call with NO retrieved context -- it can
    # only draw on generic training-data patterns, never this company's
    # actual handbook (which was never public, so never in training data).
    return "Typically, companies offer around 15 days of PTO for new employees."

def grounded_answer(question: str, retrieved: dict) -> str:
    # Stands in for an LLM call given ONLY the retrieved snippet as context.
    return f"According to {retrieved['id']} (page {retrieved['page']}): \"{retrieved['text']}\""

question = "How many vacation days does a new hire get?"
retrieved = handbook_sections[1]  # Day 2/3 show how this lookup happens automatically

print("Ungrounded:", ungrounded_answer(question))
print("Grounded:  ", grounded_answer(question, retrieved))


**The four-stage RAG flow** (see the diagram in `daily/Day1_llm-limits-and-rag-intro.md`):
documents -> index (chunk + embed) -> retrieve (top-k similarity search) -> generate
(answer constrained to retrieved text). The rest of this notebook builds each stage for
real: Day 2 builds `embed()` and the similarity math, Day 3 builds the storage/search
layer, and Day 4 builds the chunking and the final `answer()` function.


## Day 2: Turning Text Into Numbers You Can Compare

To retrieve "the relevant passage" automatically we need a numeric notion of
"how close is this question to this passage in meaning." That's an embedding
plus a similarity score. Below: a from-scratch character-trigram toy
embedding (no network calls, no trained model, just hashing), cosine
similarity computed by hand and cross-checked with numpy, and then a real,
fully runnable TF-IDF embedding via scikit-learn for comparison.


In [ ]:
import hashlib, math

def toy_embed(text: str, dims: int = 64) -> list:
    # dims=64 -> every text becomes a fixed-length "fingerprint" vector,
    # regardless of how long the original text was.
    vec = [0.0] * dims
    text = text.lower().replace(" ", "_")
    for i in range(len(text) - 2):
        trigram = text[i:i + 3]
        # Hash each 3-character window into one of 64 buckets. Different
        # trigrams CAN collide into the same bucket -- a deliberate
        # fixed-size-vs-precision trade-off, not a bug.
        bucket = int(hashlib.md5(trigram.encode()).hexdigest(), 16) % dims
        vec[bucket] += 1.0
    return vec  # -> list[float], len == dims

def cosine_similarity(a: list, b: list) -> float:
    dot = sum(x * y for x, y in zip(a, b))     # projects a onto b (unnormalized)
    norm_a = math.sqrt(sum(x * x for x in a))    # length of a
    norm_b = math.sqrt(sum(y * y for y in b))    # length of b
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

query = "how much paid time off do I get"
query_vec = toy_embed(query)
print("query vector length (dims):", len(query_vec))
print("nonzero buckets:", sum(1 for x in query_vec if x != 0), "out of", len(query_vec))

passages = [s["text"] for s in handbook_sections]
ranked = sorted(
    ((cosine_similarity(query_vec, toy_embed(doc)), doc) for doc in passages),
    reverse=True,
)
for score, doc in ranked:
    print(f"{score:.4f}  {doc}")


**Limitation to notice:** the sentence that actually answers the question --
"New hires accrue 12 vacation days..." -- ranks **last** of four. "The office
is closed on all federal holidays" ranks first purely because it shares more
3-character substrings with the query than the real answer does. The toy
embedding matches *spelling*, not *meaning* -- a real, trained embedding model
would place "paid time off" and "vacation days" close together despite
sharing zero words, because it learned they're used in the same contexts.


In [ ]:
import numpy as np

# Hand-checkable geometric intuition for cosine similarity, using simple
# 2D vectors instead of 64-dim hashed ones.
def cos_np(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

a = np.array([1.0, 0.0])   # points east
b = np.array([1.0, 1.0])   # points northeast (45 degrees from a)
c = np.array([0.0, 1.0])   # points north (90 degrees from a -- orthogonal)
d = np.array([5.0, 0.0])   # points east, SAME direction as a, but 5x longer

print("cos(a, b) =", cos_np(a, b), " expected cos(45 deg) =", math.cos(math.radians(45)))
print("cos(a, c) =", cos_np(a, c), " expected 0.0 (perpendicular = unrelated)")
print("cos(a, a) =", cos_np(a, a), " expected 1.0 (identical direction)")
print("cos(a, d) =", cos_np(a, d), " expected 1.0 (length dropped out -- direction is all that matters)")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

# TF-IDF: a real, runnable step up from character-hashing -- vectors over
# WORD vocabulary, weighted by how distinctive each word is across the
# collection. Still not a trained semantic model, but no longer matching
# arbitrary 3-character fragments either.
vectorizer = TfidfVectorizer()
doc_matrix = vectorizer.fit_transform(passages)     # shape: (4 docs, V terms), sparse
query_vec_tfidf = vectorizer.transform([query])      # shape: (1, V terms), sparse

print("vocabulary size (V):", len(vectorizer.vocabulary_))
print("doc_matrix shape:", doc_matrix.shape)

sims = sk_cosine(query_vec_tfidf, doc_matrix)[0]     # shape: (4,) one score per doc
for score, doc in sorted(zip(sims, passages), reverse=True):
    print(f"{score:.4f}  {doc}")


TF-IDF gets the top hit right ("paid time off requests" literally shares
words with the query) but the real semantic answer ("12 vacation days")
still scores a flat 0.0 -- it shares no words with "paid time off" at all.
TF-IDF fixes the *spelling* problem but not the *synonym* problem; only a
trained embedding model closes that gap.


## Day 3: Storing Embeddings at Scale -- Vector Databases

A Python loop comparing a query against every stored vector is fine for 4
passages, not for 20,000+ document chunks: it's O(n) per query. Below: a
real in-memory vector store (create/add/query, backed by one numpy matrix
instead of a proprietary index) and a measured speed comparison against a
pure-Python loop doing identical math.


In [ ]:
class InMemoryVectorStore:
    """
    Minimal stand-in for a real vector database collection: same
    create/add/query shape, but the index IS a numpy matrix, and search is
    one matrix-vector multiply instead of an ANN index walk.
    """
    def __init__(self, dims: int):
        self.dims = dims
        self.ids, self.documents, self.metadatas = [], [], []
        self.embeddings = np.zeros((0, dims), dtype=np.float64)  # shape: (n_items, dims)

    def add(self, ids, documents, embeddings, metadatas):
        assert len(ids) == len(documents) == len(embeddings) == len(metadatas)
        self.ids.extend(ids)
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        new_block = np.array(embeddings, dtype=np.float64)          # shape: (n_new, dims)
        self.embeddings = np.vstack([self.embeddings, new_block])    # shape: (n_total, dims)

    def query(self, query_embedding, n_results: int = 3):
        if not self.ids:
            return []
        q = np.asarray(query_embedding, dtype=np.float64)  # shape: (dims,)
        # One matrix-vector multiply scores every stored vector at once --
        # this IS the cosine_similarity loop above, vectorized.
        with np.errstate(all="ignore"):  # benign platform BLAS warning; verified no NaN/inf below
            dots = self.embeddings @ q                                        # shape: (n_items,)
            norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(q)  # shape: (n_items,)
            sims = np.divide(dots, norms, out=np.zeros_like(dots), where=norms != 0)  # shape: (n_items,)
        top_idx = np.argsort(-sims)[:n_results]   # shape: (n_results,), best score first
        return [
            {"id": self.ids[i], "document": self.documents[i],
             "metadata": self.metadatas[i], "score": float(sims[i])}
            for i in top_idx
        ]

collection = InMemoryVectorStore(dims=64)
collection.add(
    ids=[s["id"] for s in handbook_sections],
    documents=[s["text"] for s in handbook_sections],
    embeddings=[toy_embed(s["text"]) for s in handbook_sections],
    metadatas=[{"source": "handbook.pdf", "page": s["page"]} for s in handbook_sections],
)
print("embeddings matrix shape:", collection.embeddings.shape)

results = collection.query(toy_embed("how much PTO do new hires get"), n_results=3)
for r in results:
    print(f"{r['score']:.4f}  [{r['id']}] {r['document']}  {r['metadata']}")


Notice `sec-4.2` (the actual "12 vacation days" answer) again ranks last of
the three returned -- the vector store's `add()`/`query()` machinery is
working exactly right, it's just faithfully searching over the same weak
toy embeddings from Day 2. **A vector database makes bad embeddings fast to
search, not good** -- retrieval quality is won or lost back at the
embedding step, not the storage step.


In [ ]:
import time

# Measure the actual speedup from vectorization at a more realistic scale:
# n=20,000 random 64-dim vectors, compared via the store above vs. a plain
# Python loop doing the identical cosine-similarity math one item at a time.
rng = np.random.default_rng(0)
n = 20_000
big_store = InMemoryVectorStore(dims=64)
big_store.add(
    ids=[f"doc-{i}" for i in range(n)],
    documents=["synthetic"] * n,
    embeddings=rng.random((n, 64)),   # shape: (20000, 64)
    metadatas=[{} for _ in range(n)],
)
q = rng.random(64)

t0 = time.perf_counter()
_ = big_store.query(q, n_results=5)
t1 = time.perf_counter()
vectorized_ms = (t1 - t0) * 1000
print(f"vectorized numpy query, n={n}: {vectorized_ms:.2f} ms")

def cosine_similarity_plain(u, v):
    # Deliberately plain-Python math (not numpy per call) -- this is the
    # fair comparison: Python-level iteration doing the SAME arithmetic
    # one item at a time, vs. numpy doing it all in one vectorized call.
    dot = sum(x * y for x, y in zip(u, v))
    norm_u = math.sqrt(sum(x * x for x in u))
    norm_v = math.sqrt(sum(x * x for x in v))
    return dot / (norm_u * norm_v) if norm_u and norm_v else 0.0

t0 = time.perf_counter()
scored = sorted(
    ((cosine_similarity_plain(q, big_store.embeddings[i]), i) for i in range(n)),
    reverse=True,
)[:5]
t1 = time.perf_counter()
loop_ms = (t1 - t0) * 1000
print(f"pure python loop query, n={n}: {loop_ms:.2f} ms")
print(f"speedup: {loop_ms / vectorized_ms:.1f}x  (timing is machine-dependent -- order of magnitude is the point)")


**Distance vs. similarity:** most vector databases return a *distance*
(lower = more relevant) -- often `cosine_distance = 1 - cosine_similarity` --
the opposite direction from the similarity scores above (higher = more
relevant). Always check which convention a given library uses before
sorting; the wrong direction produces a plausible-looking but backwards
ranking with no error to flag it.

At production scale, teams typically use a managed vector database service,
a self-hosted vector search engine, or a vector extension on a database
they already run -- the create/add/query shape above stays roughly the same
across all of them.


## Day 4: Chunking Long Docs and Answering Only From Retrieved Context

A full handbook is too long (and mostly irrelevant, per question) to send on
every request. We chunk it with overlap, retrieve only the relevant chunks
for a given question via TF-IDF (Day 2) + a vector store (Day 3), and answer
strictly from them -- refusing honestly when nothing relevant comes back.


In [ ]:
def chunk_text(text: str, chunk_words: int = 40, overlap_words: int = 8) -> list:
    # Word counts track "how much meaning" a chunk holds more consistently
    # than raw character counts across documents with different sentence styles.
    words = text.split()                    # -> list[str]
    chunks, start = [], 0
    step = chunk_words - overlap_words        # window advance per iteration
    while start < len(words):
        end = start + chunk_words
        chunks.append(" ".join(words[start:end]))   # -> str, <= chunk_words words
        if end >= len(words):
            break
        start += step
    return chunks  # -> list[str]

handbook = """
Section 4.1: The office is closed on all federal holidays including New Year's Day,
Independence Day, and Thanksgiving. Employees are not required to use PTO for these days.
Section 4.2: New hires accrue 12 vacation days in their first year of employment, credited
monthly at a rate of one day per month. After three years of service the accrual rate
increases to 18 days per year.
Section 4.3: Paid time off requests must be submitted through the HR portal at least two
weeks in advance for any absence longer than two consecutive days. Same-day sick leave
does not require advance notice.
Section 5.1: Employee laptops are replaced every three years or upon failure, whichever
comes first. Submit a replacement request through the IT ticketing system.
"""

chunks = chunk_text(handbook, chunk_words=40, overlap_words=8)
print(f"chunked handbook ({len(handbook.split())} words) into {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"  chunk {i}: {len(c.split())} words")


In [ ]:
# Real end-to-end retrieval over the chunks, using the same TF-IDF approach as Day 2.
chunk_vectorizer = TfidfVectorizer()
chunk_matrix = chunk_vectorizer.fit_transform(chunks)   # shape: (n_chunks, V)
print("chunk_matrix shape:", chunk_matrix.shape)

def search(query: str, top_k: int = 2):
    q_vec = chunk_vectorizer.transform([query])                    # shape: (1, V)
    sims = sk_cosine(q_vec, chunk_matrix)[0]                        # shape: (n_chunks,)
    ranked_idx = np.argsort(-sims)[:top_k]                          # shape: (top_k,)
    return [
        {"text": chunks[i], "score": float(sims[i]), "source": f"handbook.pdf#chunk{i}"}
        for i in ranked_idx
    ]

for h in search("how many vacation days do new hires get?"):
    print(f"{h['score']:.3f}  {h['source']}  {h['text'][:70]}...")


In [ ]:
import re

def call_llm(prompt: str) -> str:
    # Stand-in for a real LLM call constrained to ONLY the given context
    # (no network access in this environment).
    return "New hires accrue 12 vacation days in their first year, credited monthly."

def answer(question: str, top_k: int = 2, min_relevance: float = 0.05) -> dict:
    hits = search(question, top_k=top_k)                     # Day 2/3 retrieval
    relevant = [h for h in hits if h["score"] >= min_relevance]

    if not relevant:
        return {"answer": "I don't know -- nothing relevant was found in the documents.", "sources": []}

    context = "\n\n".join(f"[{h['source']}] {h['text']}" for h in relevant)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the context doesn't contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    reply = call_llm(prompt)
    return {"answer": reply, "sources": [h["source"] for h in relevant]}

print("=== case 1: answerable question ===")
r1 = answer("how many vacation days do new hires get?")
print(r1)

# Cheap mechanical grounding check: every number the answer cites should
# actually appear somewhere in the retrieved context (catches a model
# inventing a DIFFERENT number while still citing a real-looking source).
context_used = " ".join(h["text"] for h in search("how many vacation days do new hires get?"))
cited_numbers = re.findall(r"\d+", r1["answer"])
print("cited numbers:", cited_numbers, "-> all present in retrieved context:", all(n in context_used for n in cited_numbers))


In [ ]:
print("=== case 2: out-of-scope question (real pitfall) ===")
r2 = answer("what is the company's parental leave policy?")
print(r2)
print()
print("The handbook never mentions parental leave, but both retrieved chunk")
print("scores clear the 0.05 threshold -- so instead of an honest 'I don't")
print("know', this returns a confident-looking answer with sources attached.")

print()
print("=== why: lexical overlap on a single word, even after removing stop words ===")
strict_vectorizer = TfidfVectorizer(stop_words="english")
strict_matrix = strict_vectorizer.fit_transform(chunks)
q_vec = strict_vectorizer.transform(["what is the company's parental leave policy?"])
sims = sk_cosine(q_vec, strict_matrix)[0]
for i, s in enumerate(sims):
    print(f"chunk {i}: {s:.3f}")
print()
print("chunk 2 alone scores nonzero -- it shares the word 'leave' with the query")
print("('same-day SICK LEAVE' vs. 'parental LEAVE policy') -- same topic word,")
print("completely different topic. This is the Day 2 spelling-vs-meaning gap")
print("resurfacing at the most dangerous point: it can flip the system from an")
print("honest refusal to a confidently-cited wrong answer.")


**Takeaway for the week:** a RAG system is only as honest as its refusal to
answer. Every stage here was necessary -- embeddings (Day 2) to make
"relevant" numeric, a vector store (Day 3) to make that search fast at
scale, and chunking with overlap (Day 4) to make long documents
retrievable without losing facts at boundaries -- but the pitfall above
shows that none of it is sufficient on its own: the relevance threshold and
the underlying embedding model both need to be tuned and tested against
real out-of-scope questions, or a confident wrong answer slips through
exactly where an honest "I don't know" was needed most.
